# 03 · Name frequency (exploration)

**Goal:** build frequency tables for the name-frequency features and check whether they explain false alarms.

**Approach:** in the OpenSanctions targets export (`targets.simple`), count once per entity, separately for persons and for companies/organisations:
- **words** ("thomas", "muller") — section 2
- **full names** ("muller thomas", word order ignored) — section 2b

**Status:** exploration only. Nothing is written to `data/`; the tables live as variables in this notebook. Once we are happy, the code moves to `src/` and `scripts/`.

Runtime: about 4–6 minutes for the whole notebook. Kernel: *Python (ds_sanctions)*.

In [1]:
import sys, re, time
sys.path.insert(0, "../src")

from collections import Counter
import numpy as np
import pandas as pd
from unidecode import unidecode
from screening import config as C

TARGETS_CSV = C.DATA_RAW / "targets.simple-20261111.csv"   # downloaded 2026-11-11
GROUPS = {
    "person": ("Person",),
    "org": ("Company", "Organization", "LegalEntity"),
}
MIN_LEN = 2   # drop one-letter words (initials)

targets = pd.read_csv(TARGETS_CSV, usecols=["id", "schema", "name", "aliases"],
                      dtype=str, keep_default_na=False)
print(f"{len(targets):,} target entities")
targets["schema"].value_counts()

1,248,755 target entities


schema
Person          1036156
Company          100010
LegalEntity       58888
Security          20568
CryptoWallet      13812
Organization       9755
Vessel             9164
Airplane            342
Asset                36
Address              19
PublicBody            4
Position              1
Name: count, dtype: int64

## 1 · Normalisation

Every name is transliterated to ASCII (`unidecode`), lowercased, and everything that is not a letter or digit becomes a space. The **same function** must later be used for the pairs, otherwise "müller" in a pair would not find "muller" in the table.

In [2]:
_NON_ALNUM = re.compile(r"[^a-z0-9]+")

def normalize(name):
    if not name:
        return ""
    return " ".join(_NON_ALNUM.sub(" ", unidecode(name).lower()).split())

def tokens(name, min_len=MIN_LEN):
    return [t for t in normalize(name).split() if len(t) >= min_len]

def entity_tokens(name, aliases, min_len=MIN_LEN):
    """Set of words over the name and all ';'-separated aliases of one entity."""
    names = [name] + ([a for a in aliases.split(";") if a.strip()] if aliases else [])
    return {t for n in names for t in tokens(n, min_len)}

examples = ["Thomas Müller", "THOMAS MULLER", "Олександр Петренко", "عبد الكريم",
            "王伟", "J. Smith", "Myanmar Yatai International Holding Group Co., Ltd.",
            'ООО "Газпром добыча Ноябрьск"']
pd.DataFrame({"original": examples, "normalized": [normalize(e) for e in examples]})

,original,normalized
0,Thomas Müller,thomas muller
1,THOMAS MULLER,thomas muller
2,Олександр Петренко,oleksandr petrenko
3,عبد الكريم,bd lkrym
4,王伟,wang wei
5,J. Smith,j smith
6,"Myanmar Yatai International Holding Group Co.,...",myanmar yatai international holding group co ltd
7,"ООО ""Газпром добыча Ноябрьск""",ooo gazprom dobycha noiabr sk


## 2 · Count words once per entity

For each entity: the **set** of its words across name and aliases. Then count in how many entities each word appears.

- 5 different people called Thomas Müller → "thomas" +5
- 1 person with 5 spellings → each word +1 only

Result per group: a table with `token`, `n_entities` (number of entities whose names contain the word) and `share` (n_entities / entities in the group).

In [4]:
def count_tokens(df):
    counts = Counter()
    for name, aliases in zip(df["name"], df["aliases"]):
        counts.update(entity_tokens(name, aliases))
    return counts

freq, n_entities = {}, {}
for group, schemas in GROUPS.items():
    sub = targets[targets["schema"].isin(schemas)]
    t0 = time.time()
    counts = count_tokens(sub)
    table = (pd.DataFrame(list(counts.items()), columns=["token", "n_entities"])
               .sort_values("n_entities", ascending=False, ignore_index=True))
    table["share"] = table["n_entities"] / len(sub)
    freq[group], n_entities[group] = table, len(sub)
    print(f"{group:6s} {len(sub):>10,} entities  {len(table):>10,} distinct words  ({time.time() - t0:.0f}s)")

lookup = {g: dict(zip(t["token"], t["n_entities"])) for g, t in freq.items()}
freq["person"].head(10)

person  1,036,156 entities     713,706 distinct words  (64s)
org       168,653 entities     152,820 distinct words  (19s)


,token,n_entities,share
0,de,45316,0.043735
1,jose,22862,0.022064
2,maria,21863,0.021100
3,da,20661,0.019940
4,silva,17763,0.017143
5,jean,13613,0.013138
6,li,12039,0.011619
7,antonio,10857,0.010478
8,santos,10400,0.010037
9,er,10338,0.009977


## 2b · Count full names once per entity

Same idea for the whole name. The key is the **sorted** normalised words, so "Thomas Müller", "MÜLLER, Thomas" and "Muller Thomas" are one name: `muller thomas`. Here one-letter words are kept (min length 1), so "J. Smith" and "John Smith" stay different names.

Result per group: `full_name`, `n_entities` (number of entities carrying this name or alias).

In [5]:
def name_key(name):
    """Order-insensitive key of one full name; '' if nothing is left."""
    return " ".join(sorted(tokens(name, min_len=1)))

def entity_name_keys(name, aliases):
    names = [name] + ([a for a in aliases.split(";") if a.strip()] if aliases else [])
    return {k for k in (name_key(n) for n in names) if k}

full_freq = {}
for group, schemas in GROUPS.items():
    sub = targets[targets["schema"].isin(schemas)]
    t0 = time.time()
    counts = Counter()
    for name, aliases in zip(sub["name"], sub["aliases"]):
        counts.update(entity_name_keys(name, aliases))
    table = (pd.DataFrame(list(counts.items()), columns=["full_name", "n_entities"])
               .sort_values("n_entities", ascending=False, ignore_index=True))
    full_freq[group] = table
    shared = (table["n_entities"] >= 2).mean()
    print(f"{group:6s} {len(table):>10,} distinct full names, {shared:.1%} carried by 2+ entities  ({time.time() - t0:.0f}s)")

full_lookup = {g: dict(zip(t["full_name"], t["n_entities"])) for g, t in full_freq.items()}

display(pd.concat({g: t.head(25).reset_index(drop=True) for g, t in full_freq.items()}, axis=1))

name_bands = [0, 2, 5, 20, 10**7]
name_labels = ["1-2", "3-5", "6-20", ">20"]
display(pd.concat({g: pd.cut(t["n_entities"], name_bands, labels=name_labels).value_counts().sort_index()
                   for g, t in full_freq.items()}, axis=1))

probe_names = ["Thomas Müller", "Ivan Ivanov", "David Smith", "John Smith", "Abdul Karim",
               "Abdul Kareem", "Muhammad Ali", "Li Wei", "Ilya Eduardovich Torosov"]
pd.DataFrame({"name": probe_names, "key": [name_key(n) for n in probe_names],
              "n_entities": [full_lookup["person"].get(name_key(n), 0) for n in probe_names]})

person  1,845,118 distinct full names, 5.0% carried by 2+ entities  (73s)
org       294,561 distinct full names, 1.3% carried by 2+ entities  (20s)


person             \
                full_name n_entities   
0        austestad morten        109   
1                  person         94   
2   jorn seljelid steinar         77   
3                   bw mr         65   
4                abu omar         58   
5                    siwy         48   
6               abu aisha         46   
7                 bw sh y         45   
8                   gruby         42   
9            abdullah abu         41   
10              bd bw llh         37   
11                abu ali         37   
12                   lysy         36   
13              abu musab         34   
14              abu ahmad         34   
15               abu anas         34   
16                 bw hmd         34   
17               muhammad         33   
18            abu khattab         33   
19                bw khtb         32   
20                b bw ms         32   
21              abu hamza         32   
22                  bw ly         31   
23                 bw hjr         30   
24              abu hajar         30   

                                                  org             
                                            full_name n_entities  
0                                 bank first national         28  
1                 federatsii pravitel rossiiskoi stvo         26  
2                            aktsionernoe obshchestvo         23  
3                                                  ao         19  
4   al ians iu obshchestvo ogranichennoi otvetstve...         18  
5                 aktsionernoe obshchestvo publichnoe         18  
6   avant limited nistiu obmezhenoiu tovaristvo tr...         17  
7                                   sasanian shahryar         16  
8                                    adamos montanios         15  
9   del iu obshchestvo ogranichennoi otvetstvennos...         15  
10  iu obshchestvo ogranichennoi orion otvetstvenn...         14  
11                                nqlb psdrn slmy sph         13  
12  iu obshchestvo ogranichennoi otvetstvennost pe...         13  
13                       gosudarstvennoe uchrezhdenie         13  
14  iu obshchestvo ogranichennoi otvetstvennost pr...         12  
15                                                ooo         12  
16                       goodgate limited secretarial         12  
17                                     kushnir mykola         11  
18  iu novye obshchestvo ogranichennoi otvetstvenn...         11  
19  avangard iu obshchestvo ogranichennoi otvetstv...         11  
20                            elene papapaulou pastou         11  
21  iu obshchestvo ogranichennoi otvetstvennost s ...         11  
22                      brovkova mikhailivna svitlana         10  
23  iu obshchestvo ogranichennoi omega otvetstvenn...         10  
24                     associates medicine preventive         10

,person,org
n_entities,,
1-2,1834514,293992
3-5,9585,477
6-20,974,89
>20,45,3


,name,key,n_entities
0,Thomas Müller,muller thomas,2
1,Ivan Ivanov,ivan ivanov,26
2,David Smith,david smith,18
3,John Smith,john smith,8
4,Abdul Karim,abdul karim,8
5,Abdul Kareem,abdul kareem,7
6,Muhammad Ali,ali muhammad,18
7,Li Wei,li wei,11
8,Ilya Eduardovich Torosov,eduardovich ilya torosov,1


## 3 · What the tables look like

In [6]:
# most common words per group
pd.concat({g: t.head(30).reset_index(drop=True) for g, t in freq.items()}, axis=1)

person                                   org                     
        token n_entities     share            token n_entities     share
0          de      45316  0.043735      obshchestvo      34586  0.205072
1        jose      22862  0.022064               iu      28108  0.166662
2       maria      21863  0.021100    ogranichennoi      28057  0.166359
3          da      20661  0.019940   otvetstvennost      28046  0.166294
4       silva      17763  0.017143              ltd      15472  0.091739
5        jean      13613  0.013138             xian      13491  0.079993
6          li      12039  0.011619          company      12088  0.071674
7     antonio      10857  0.010478               co      10854  0.064357
8      santos      10400  0.010037          limited      10795  0.064007
9          er      10338  0.009977               si       9837  0.058327
10         al      10272  0.009914             gong       9757  0.057853
11         si      10255  0.009897              shi       9079  0.053832
12      david       8731  0.008426              noe       9066  0.053755
13     carlos       8598  0.008298               of       8963  0.053145
14         ke       8495  0.008199              you       8928  0.052937
15      evich       8395  0.008102              xin       8383  0.049706
16     garcia       8293  0.008004     uchrezhdenie       8355  0.049540
17       mhmd       7890  0.007615               in       8077  0.047891
18        dos       7692  0.007424             ltda       7660  0.045419
19       john       7654  0.007387            jiang       7657  0.045401
20         la       7646  0.007379              llc       7343  0.043539
21   oliveira       7612  0.007346       tovaristvo       7324  0.043426
22      marie       7350  0.007094       tovarystvo       7120  0.042217
23  francisco       7130  0.006881              inc       7058  0.041849
24    michael       6734  0.006499         xinjiang       6844  0.040580
25     daniel       6731  0.006496  gosudarstvennoe       6823  0.040456
26     robert       6577  0.006347            kuang       6627  0.039294
27       luis       6278  0.006059     aktsionernoe       6509  0.038594
28      james       5954  0.005746              and       6156  0.036501
29         fu       5798  0.005596               li       6150  0.036465

In [7]:
# how many words fall into each frequency band
bands = [0, 1, 2, 10, 100, 1000, 10**7]
labels = ["1", "2", "3-10", "11-100", "101-1000", ">1000"]
pd.concat({g: pd.cut(t["n_entities"], bands, labels=labels).value_counts().sort_index()
           for g, t in freq.items()}, axis=1)

,person,org
n_entities,,
1,477032,103067
2,98727,19733
3-10,104028,21687
11-100,29616,7139
101-1000,3835,1007
>1000,468,187


In [8]:
# probe words
probe_person = ["thomas", "muller", "abdul", "kareem", "karim", "ali", "muhammad",
                "ivanov", "petrenko", "torosov", "smith", "wang", "li", "kim"]
probe_org = ["ltd", "llc", "ooo", "obshchestvo", "gmbh", "company", "bank", "gazprom"]
display(pd.DataFrame({"token": probe_person, "n_entities": [lookup["person"].get(t, 0) for t in probe_person]}))
display(pd.DataFrame({"token": probe_org, "n_entities": [lookup["org"].get(t, 0) for t in probe_org]}))

,token,n_entities
0,thomas,4593
1,muller,409
2,abdul,2539
3,kareem,84
4,karim,386
5,ali,4929
6,muhammad,5214
7,ivanov,738
8,petrenko,31
9,torosov,1


,token,n_entities
0,ltd,15472
1,llc,7343
2,ooo,2917
3,obshchestvo,34586
4,gmbh,453
5,company,12088
6,bank,2981
7,gazprom,275


## 4 · Cross-check with names.txt

`names.txt` counts *spellings* (each distinct name string once), not entities, and mixes all schemas. If both sources rank words similarly, the feature is robust to the choice of source. Here each word is counted once per line of names.txt.

In [9]:
with open(C.NAMES_TXT, encoding="utf-8") as f:
    names_txt = Counter(t for line in f for t in set(tokens(line.rstrip("\n"))))

all_entities = pd.concat(freq.values()).groupby("token", as_index=False)["n_entities"].sum()
cmp = all_entities.merge(pd.DataFrame(list(names_txt.items()), columns=["token", "n_names_txt"]),
                         on="token", how="inner")
print(f"words in both sources: {len(cmp):,}")
print("Spearman rank correlation:", round(cmp["n_entities"].corr(cmp["n_names_txt"], method="spearman"), 3))
cmp.assign(spellings_per_entity=cmp["n_names_txt"] / cmp["n_entities"]).sort_values("n_entities", ascending=False).head(20)

words in both sources: 820,999
Spearman rank correlation: 0.769


,token,n_entities,n_names_txt,spellings_per_entity
160951,de,49530,75887,1.532142
526439,obshchestvo,34586,99223,2.868878
310340,iu,28450,77513,2.724534
528844,ogranichennoi,28057,76937,2.742168
540483,otvetstvennost,28046,76444,2.725665
326235,jose,22997,26358,1.146149
153147,da,22261,28828,1.295000
444118,maria,22006,27692,1.258384
664862,si,20092,89861,4.472477
411018,li,18189,33764,1.856287


## 5 · Does it explain false alarms?

For each pair, two values:
- **rarest shared word:** frequency of the rarest word both sides share. A shared rare word ("torosov") is strong evidence; a shared common word ("ali") is weak.
- **rarest shared full name:** frequency of the rarest full name (order-insensitive key) both sides share; "none" if they share no full name.

⚠️ Runs on the **uncleaned** `pairs_scoped.parquet`: self-pairs inflate the positives. Re-run after the cleaning.
⚠️ `targets.simple` is the state *after* OpenSanctions merged duplicates: a name shared by a negative pair counts ≥ 2, by a positive pair ≥ 1. That is why 1 and 2 share one band below. The effect is strongest for full names.

In [ ]:
pairs = pd.read_parquet(C.PAIRS_SCOPED, columns=["pair_id", "judgement", "left_schema",
                                                 "left_name", "left_alias", "right_name", "right_alias"])

def side_tokens(names, aliases):
    return {t for n in list(names) + list(aliases) for t in tokens(n)}

def side_keys(names, aliases):
    return {k for k in (name_key(n) for n in list(names) + list(aliases)) if k}

t0 = time.time()
rows = []
for r in pairs.itertuples(index=False):
    group = "person" if r.left_schema == "Person" else "org"
    shared_words = side_tokens(r.left_name, r.left_alias) & side_tokens(r.right_name, r.right_alias)
    shared_names = side_keys(r.left_name, r.left_alias) & side_keys(r.right_name, r.right_alias)
    rarest_word = min((lookup[group].get(t, 0) for t in shared_words), default=np.nan)
    rarest_name = min((full_lookup[group].get(k, 0) for k in shared_names), default=np.nan)
    rows.append((r.pair_id, r.judgement, group == "person",
                 len(shared_words), rarest_word, len(shared_names), rarest_name))
res = pd.DataFrame(rows, columns=["pair_id", "judgement", "person",
                                  "n_shared_words", "rarest_word", "n_shared_names", "rarest_name"])
print(f"{len(res):,} pairs in {time.time() - t0:.0f}s")
print(f"no shared word: {res['rarest_word'].isna().mean():.1%}   no shared full name: {res['rarest_name'].isna().mean():.1%}")

In [ ]:
word_bands = [-1, 0, 2, 10, 100, 1000, 10**7]
word_labels = ["0", "1-2", "3-10", "11-100", "101-1000", ">1000"]
res["word_band"] = pd.cut(res["rarest_word"], word_bands, labels=word_labels)

full_bands = [-1, 0, 2, 5, 20, 10**7]
full_labels = ["0", "1-2", "3-5", "6-20", ">20"]
res["name_band"] = (pd.cut(res["rarest_name"], full_bands, labels=full_labels)
                      .cat.add_categories("none").fillna("none"))

def neg_share_table(s, col):
    tab = pd.crosstab(s[col], s["judgement"])
    tab["neg_share"] = (tab["negative"] / tab.sum(axis=1)).round(3)
    return tab

for person, title in [(True, "PERSON"), (False, "COMPANY / ORG")]:
    s = res[res["person"] == person]
    print(f"{title} · by rarest shared WORD"); display(neg_share_table(s, "word_band"))
    print(f"{title} · by rarest shared FULL NAME"); display(neg_share_table(s, "name_band"))

### Does the full name add something beyond the rarest word?

Share of negative pairs for each combination of word band (rows) and full-name band (columns), persons only. If the share changes across a row, the full name explains false alarms that the word frequency alone does not.

In [ ]:
s = res[res["person"]]
counts = pd.crosstab(s["word_band"], s["name_band"])
neg = pd.crosstab(s["word_band"], s["name_band"], values=(s["judgement"] == "negative"), aggfunc="mean").round(3)
print("number of pairs"); display(counts)
print("share negative"); display(neg)

## Open questions

- Stop list for company words (ltd, llc, ooo …) — needed, or does "rarest shared word" already ignore them?
- Features as `log1p` or as bands? Bands absorb the ±1 merge effect; for full names bands seem necessary.
- Keep both features (rarest word, rarest full name), or is one enough? See the combination table.
- Mean over shared words as an extra feature?
- Re-run section 5 after the cleaning.